# Lab 1B — Refactor the Legacy Script · ✅ SOLUTION

**Week 1 · Day 1 · AI Engineering Academy — the anchor lab that closes Day 1**

> 👩‍🏫 Fully worked, including both stretch goals and the Part 3 day-closer. All data is synthetic; Part 3 uses a deterministic stub, not a real model call.
>
> 📌 Cohort standard is **Python 3.13**. The `list[Order]` / `dict[str, int]` generic syntax requires Python 3.9+, so it is safe across the cohort environment.

| Segment | Focus | ~Time |
|---|---|---|
| Part 0 | Read the legacy script, run the baseline | 5 min |
| Part 1 | Refactor: constants, dataclass, parse, transforms, smoke test | 35–45 min |
| Part 2 | Validation, custom exception, docstrings (stretch) | 15–20 min |
| Part 3 | Pull the day together: config object + model stub | 10–15 min |
| Close | Restart & Run All + checks for understanding | 5 min |

**Total: ~70–90 minutes.**

## Part 0 — The starting point (the legacy baseline)

In [ ]:
# Synthetic Cordwell Home & Hardware contractor orders (fictional data).
# This is the raw input both the legacy script and your refactor will read.
RAW_ORDERS: list[dict] = [
    {"id": "ORD-1001", "amount": 150.00,  "status": "shipped",   "customer": "Ridgeline Builders"},
    {"id": "ORD-1002", "amount": 42.50,   "status": "pending",   "customer": "Hammersmith Contracting"},
    {"id": "ORD-1003", "amount": 980.00,  "status": "shipped",   "customer": "Oakfield Renovations"},
    {"id": "ORD-1004", "amount": 18.99,   "status": "shipped",   "customer": "DIY Walk-in"},
    {"id": "ORD-1005", "amount": 305.75,  "status": "cancelled", "customer": "Maple & Stone LLC"},
    {"id": "ORD-1006", "amount": 615.00,  "status": "shipped",   "customer": "Ironclad Decks"},
    {"id": "ORD-1007", "amount": 74.20,   "status": "returned",  "customer": "Hammersmith Contracting"},
    {"id": "ORD-1008", "amount": 1240.00, "status": "shipped",   "customer": "Summit Property Group"},
    {"id": "ORD-1009", "amount": 99.99,   "status": "pending",   "customer": "DIY Walk-in"},
    {"id": "ORD-1010", "amount": 250.00,  "status": "shipped",   "customer": "Ridgeline Builders"},
    {"id": "ORD-1011", "amount": 530.00,  "status": "shipped",   "customer": "Greenfield Landscaping"},
    {"id": "ORD-1012", "amount": 12.00,   "status": "cancelled", "customer": "DIY Walk-in"},
    {"id": "ORD-1013", "amount": 460.00,  "status": "pending",   "customer": "Oakfield Renovations"},
    {"id": "ORD-1014", "amount": 88.50,   "status": "shipped",   "customer": "Maple & Stone LLC"},
]

In [ ]:
# legacy_report.py  --  DO NOT SHIP THIS
# Working but ugly: copy-paste loops, a bare except, magic numbers, zero type hints.
# It runs and prints a baseline. Your refactor must reproduce these numbers exactly.

total = 0
for o in RAW_ORDERS:
    try:
        if o["status"] == "shipped":
            total = total + o["amount"]
    except:
        pass  # <- bare except silently swallows EVERYTHING
legacy_total = total
print("Total shipped: " + str(legacy_total))

# copy-paste block #1 -- magic number 100
flagged = []
for o in RAW_ORDERS:
    if o["amount"] > 100:
        flagged.append(o["id"])
legacy_flagged = flagged
print("High-value order IDs: " + str(legacy_flagged))

# copy-paste block #2 -- manual accumulation
counts = {}
for o in RAW_ORDERS:
    s = o["status"]
    if s in counts:
        counts[s] = counts[s] + 1
    else:
        counts[s] = 1
legacy_counts = counts
print("Counts by status: " + str(legacy_counts))

# copy-paste block #3 -- nested if + another magic number 500
priority = []
for o in RAW_ORDERS:
    if o["status"] == "shipped":
        if o["amount"] > 500:
            priority.append(o["id"])
legacy_priority = priority
print("Priority review IDs: " + str(legacy_priority))

**Expected baseline:** `Total shipped: 3872.49` · 8 high-value IDs · `{'shipped': 8, 'pending': 3, 'cancelled': 2, 'returned': 1}` · 4 priority IDs. The refactor must reproduce all of these.

## Part 1 — Refactor

In [ ]:
from dataclasses import dataclass
from collections import Counter
import math

### 1.1 — Constants (Task 05)

In [ ]:
MIN_VALID_AMOUNT: float = 0.0
HIGH_VALUE_THRESHOLD: float = 100.0
PRIORITY_REVIEW_THRESHOLD: float = 500.0
SHIPPED: str = "shipped"

### 1.2 — The `Order` dataclass (Task 02)
*Teaching note:* the legacy carried each order as a bare `dict`. The dataclass gives type hints on every field, IDE autocomplete, and a free `__repr__`/`__eq__` — and it makes `Order(**record)` the natural validation boundary.

In [ ]:
@dataclass
class Order:
    id: str
    amount: float
    status: str
    customer: str

### 1.3 — `parse_orders`: the validation boundary (Tasks 03 + 04)
This is the answer to *"replace the bare except."* We don't catch anything here — we **construct typed objects at the boundary**. A malformed record raises `TypeError` loudly at parse time (missing/extra key), instead of being swallowed mid-loop. Downstream functions then operate on guaranteed-complete `Order` objects and need no defensive `try/except` at all.

In [ ]:
def parse_orders(raw: list[dict]) -> list[Order]:
    """Build typed Order objects from raw dicts.

    The single validation boundary: Order(**record) raises loudly if a record
    is missing a field or carries an unexpected one.

    Args:
        raw: Raw order dicts as received from upstream.

    Returns:
        A list of validated Order objects.
    """
    return [Order(**record) for record in raw]

### 1.4 — Transform functions (Tasks 01 + 04)

In [ ]:
def total_shipped(orders: list[Order]) -> float:
    """Sum the amounts of all shipped orders."""
    return sum(o.amount for o in orders if o.status == SHIPPED)


def flag_high_value(orders: list[Order], threshold: float = HIGH_VALUE_THRESHOLD) -> list[str]:
    """Return the IDs of orders whose amount exceeds `threshold`."""
    return [o.id for o in orders if o.amount > threshold]


def count_by_status(orders: list[Order]) -> dict[str, int]:
    """Return a mapping of status -> number of orders with that status."""
    return dict(Counter(o.status for o in orders))


def priority_review(orders: list[Order], threshold: float = PRIORITY_REVIEW_THRESHOLD) -> list[str]:
    """Return IDs of shipped orders above the priority threshold."""
    return [o.id for o in orders if o.status == SHIPPED and o.amount > threshold]

### 1.5 — Smoke test (*Done when*)

In [ ]:
orders = parse_orders(RAW_ORDERS)

assert math.isclose(total_shipped(orders), legacy_total), "total_shipped mismatch"
assert flag_high_value(orders) == legacy_flagged, "flag_high_value mismatch"
assert count_by_status(orders) == legacy_counts, "count_by_status mismatch"
assert priority_review(orders) == legacy_priority, "priority_review mismatch"
print("\u2705 Smoke check passed -- refactor matches the legacy output.")
print(f"  total_shipped       = {total_shipped(orders):,.2f}")
print(f"  flag_high_value     = {flag_high_value(orders)}")
print(f"  count_by_status     = {count_by_status(orders)}")
print(f"  priority_review     = {priority_review(orders)}")

## Part 2 — Validation & stretch
Note the distinction this makes concrete: **structural** problems (missing/extra field) fail at `parse_orders`; **semantic** problems (negative amount, empty status) are caught by `validate_order`. Two different failure modes, both *loud* — neither swallowed.

In [ ]:
class InvalidOrderError(Exception):
    """Raised when an order is structurally valid but breaks a business rule."""
    def __init__(self, order_id: str, reason: str) -> None:
        self.order_id = order_id
        super().__init__(f"Order {order_id!r} is invalid: {reason}")


def validate_order(order: Order) -> None:
    """Raise InvalidOrderError if the order fails a business rule.

    Rules:
        * amount must be >= MIN_VALID_AMOUNT
        * status must be non-empty
    """
    if order.amount < MIN_VALID_AMOUNT:
        raise InvalidOrderError(order.id, f"amount {order.amount} is below the minimum {MIN_VALID_AMOUNT}")
    if not order.status:
        raise InvalidOrderError(order.id, "status is empty")

### 2.1 — Prove it rejects bad data

In [ ]:
dirty = [
    Order(id="ORD-9001", amount=-5.0, status="shipped", customer="Phantom Co"),  # negative
    Order(id="ORD-9002", amount=10.0, status="",        customer="Mystery LLC"), # empty status
]
for bad in dirty:
    try:
        validate_order(bad)
        print(f"{bad.id}: passed (unexpected!)")
    except InvalidOrderError as e:
        print(f"Rejected loudly -> {e}")

## Part 3 — Pull the day together 🧩

The whole day in one runnable cell-block: the **config object** and **client stub** from the morning, fed by the **typed data**, **clean functions**, **named constants**, and a **comprehension** from this lab — ending with a model in the loop. The model ID stays a placeholder we never hardcode.

In [ ]:
@dataclass
class LLMConfig:
    model: str = "placeholder-model-id"  # real ID confirmed in Week 3 -- never hardcode
    temperature: float = 0.7
    max_tokens: int = 512
    timeout: float = 30.0


def call_model(prompt: str, cfg: LLMConfig) -> str:
    """Deterministic STUB. The real API call lands in Week 3."""
    return f"[STUB] model={cfg.model} | temp={cfg.temperature} | prompt_chars={len(prompt)}"

In [ ]:
def build_review_prompt(orders: list[Order]) -> str:
    """Pure function: turn the priority-review orders into a prompt for the model."""
    priority_ids = set(priority_review(orders))
    lines = [
        f"- {o.id}: ${o.amount:,.2f} ({o.customer})"
        for o in orders
        if o.id in priority_ids
    ]
    body = "\n".join(lines) if lines else "(no priority orders)"
    return (
        "Draft a one-paragraph internal note flagging these high-value shipped "
        "orders for finance review:\n" + body
    )

In [ ]:
cfg = LLMConfig()
prompt = build_review_prompt(orders)
response = call_model(prompt, cfg)

print("--- prompt sent to the model -------------------------")
print(prompt)
print("--- model response (stubbed) -------------------------")
print(response)

# Responsible-AI gesture: even a stub gets a basic sanity check before we trust it.
assert response, "model returned an empty response"
print("\n\u2705 Day pulled together: typed data -> clean functions -> a model in the loop.")

### Responsible-AI note
That one-line `assert response` is a placeholder for real **output validation**. When `call_model` becomes a live call in Week 3, the same seam is where you'll check the response is non-empty, parse the structure you expect, and (by Week 9) run guardrails that classify and reject unsafe or off-task output. Quality and safety checks live at the model boundary, by design — not bolted on later.

## Where today's patterns go next

- **`LLMConfig` + `call_model`** → Week 3 real call · Week 5 fine-tuned model ID via config · Week 7 behind a FastAPI endpoint · Week 8 containerized with retry logic.
- **`InvalidOrderError` / specific exceptions** → Week 8 retry logic that separates transient from fatal errors · Week 9 guardrails over model output.
- **`parse_orders` boundary + typed `Order`** → Week 2's schema validation and profiling on the curated corpus.
- **Restart-and-Run-All** → Week 7's CI gate that proves notebooks execute cleanly before code is promoted to modules.

## 👩‍🏫 Notes — likely pitfalls

From the solution walkthrough — good discussion openers when circulating:

- **Overly broad `except Exception`.** Many learners just swap the bare `except` for `except Exception` and call it done. Push back: *what specific exception can this code actually raise?* A `KeyError` on a missing dict key is a **caller bug**, not a runtime condition to catch — the right fix is the `parse_orders` boundary, which makes the `try/except` unnecessary entirely.
- **Dataclass methods that should be functions.** If a method on `Order` doesn't use `self`, it should be a standalone function. Dataclasses are data containers, not service classes. (All four transforms here are correctly free functions.)
- **Comprehension for side effects.** `[print(o) for o in orders]` is a loop wearing a comprehension costume — it builds a throwaway list of `None`. Comprehensions build collections; use a plain `for` loop for side effects.
- **Filtering before vs. after validation.** `total_shipped` trusts every `Order` is well-formed — which is only safe *because* `parse_orders` validated structure first. Make the dependency explicit when you debrief.

**Checks for understanding (expected answers):**
1. *Restart-and-Run-All protects you from* hidden kernel state — variables from deleted/edited cells that linger until restart, making a broken notebook look reproducible.
2. *A class beats a function* when it must hold state across calls (a client managing a connection); it's *overkill* for a pure data transform, which needs only a function.
3. *Config-as-object* gives one source of truth, testability (pass a stub config), discoverability (autocomplete), and currency (swap a deprecated model in one place, not via grep across string literals).